# Build a Digit Recognizer From Scratch

This notebook is the hands-on exercise from the workshop slides. Every function below—`relu`, `softmax`, `SimpleNet`, `backprop`, the training loop, the prediction step—is the *exact same code* you just saw on the slides, extended only as much as needed to actually run on real data.

No PyTorch, no TensorFlow, no autograd. Just NumPy arrays and the math we walked through: a neuron weighs its evidence, layers stack neurons, training nudges the weights downhill.

**Rules we're following:**
- We carve out a genuine **validation set** from the training data to check progress — the **test set** is touched exactly once, at the very end. (Peeking at the test set while training is how you fool yourself into thinking a model is better than it is.)
- Every cell should run top to bottom without surprises.

In [1]:
#@title Setup
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
print("Ready!")

ModuleNotFoundError: No module named 'matplotlib'

## 1. Load the digit dataset

MNIST: 70,000 handwritten digits, each a 28×28 grayscale image labeled 0–9.

We use `torchvision` **only** to download and cache the raw files — it's a convenient, reliable mirror of the dataset. Everything from this point on is plain NumPy; we never touch `torch.nn` or autograd.

In [ ]:
#@title 1. Load the MNIST dataset
import torchvision

mnist_train = torchvision.datasets.MNIST('./data', train=True, download=True)
mnist_test = torchvision.datasets.MNIST('./data', train=False, download=True)

# Flatten each 28x28 image into a 784-length vector, scale pixels from [0, 255] to [0, 1]
X_train_full = mnist_train.data.numpy().reshape(-1, 784).astype(np.float64) / 255.0
y_train_full = mnist_train.targets.numpy().astype(np.int64)

X_test = mnist_test.data.numpy().reshape(-1, 784).astype(np.float64) / 255.0
y_test = mnist_test.targets.numpy().astype(np.int64)

# Carve out a real validation set from the training data.
# We will not look at X_test / y_test again until the very last cell.
val_size = 5000
X_train, y_train = X_train_full[:-val_size], y_train_full[:-val_size]
X_val, y_val = X_train_full[-val_size:], y_train_full[-val_size:]

print(f"Training samples:   {len(X_train)}")
print(f"Validation samples: {len(X_val)}  (used to check progress while training)")
print(f"Test samples:       {len(X_test)}  (touched once, at the very end)")

## 2. A quick look at the data

Just like the slides showed — each image really is just a grid of numbers.

In [ ]:
#@title 2. Visualize a few samples
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i].reshape(28, 28), cmap='gray')
    ax.set_title(f"label: {y_train[i]}")
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. The building blocks

These are the exact functions from `relu.py`, `softmax.py`, and `loss.py` in the slides — written here to work on a whole batch of images at once instead of one number at a time.

In [ ]:
#@title 3. relu, softmax, and the loss function
def random_matrix(rows, cols):
    # Small random starting weights, scaled for ReLU layers ("He initialization").
    return np.random.randn(rows, cols) * np.sqrt(2.0 / rows)

def relu(x):
    return np.maximum(0, x)  # same idea as relu.py: keep positive signals, zero out the rest

def softmax(x):
    # subtracting the row max first is just for numerical stability - it doesn't change the result
    shifted = x - x.max(axis=1, keepdims=True)
    exps = np.exp(shifted)
    return exps / exps.sum(axis=1, keepdims=True)

def cross_entropy_loss(probs, labels):
    # same formula as loss.py: -log(confidence in the true answer), averaged over the batch
    n = len(labels)
    confidence_in_right_answer = probs[np.arange(n), labels]
    return -np.mean(np.log(confidence_in_right_answer + 1e-12))

## 4. The network

This is `simple_net.py` from the slides — 784 pixels in, a hidden layer of 128 neurons, 10 output scores — extended just enough to remember its own forward pass so it can learn from its mistakes.

In [ ]:
#@title 4. SimpleNet + backprop
class SimpleNet:
    def __init__(self):
        self.W1 = random_matrix(784, 128)
        self.W2 = random_matrix(128, 10)

    def forward(self, x):
        self.x = x
        self.z1 = x @ self.W1
        self.h = relu(self.z1)
        self.z2 = self.h @ self.W2
        self.probs = softmax(self.z2)
        return self.probs

    def update(self, grads, learning_rate):
        dW1, dW2 = grads
        self.W1 -= learning_rate * dW1
        self.W2 -= learning_rate * dW2


def backprop(net, predictions, labels):
    """Works backward from the loss to figure out how to nudge W1 and W2."""
    n = labels.shape[0]
    one_hot = np.zeros_like(predictions)
    one_hot[np.arange(n), labels] = 1

    dz2 = (predictions - one_hot) / n     # how wrong the output layer was
    dW2 = net.h.T @ dz2

    dh = dz2 @ net.W2.T
    dz1 = dh * (net.z1 > 0)               # ReLU's gradient: 1 where it fired, 0 where it didn't
    dW1 = net.x.T @ dz1

    return dW1, dW2

## 5. Train: predict, measure, adjust — thousands of times

This is `train.py` from the slides, with two additions: it works through the data in small batches instead of all 55,000 images at once, and it checks **validation** accuracy every epoch — the test set stays untouched.

In [ ]:
#@title 5. Training loop
net = SimpleNet()

learning_rate = 0.3
epochs = 20
batch_size = 128
n_batches = len(X_train) // batch_size

train_losses, val_accs = [], []

for epoch in range(epochs):
    permutation = np.random.permutation(len(X_train))
    epoch_loss = 0.0

    for i in range(n_batches):
        batch_idx = permutation[i*batch_size : (i+1)*batch_size]
        X_batch, y_batch = X_train[batch_idx], y_train[batch_idx]

        predictions = net.forward(X_batch)
        epoch_loss += cross_entropy_loss(predictions, y_batch)

        grads = backprop(net, predictions, y_batch)
        net.update(grads, learning_rate)

    epoch_loss /= n_batches

    val_predictions = net.forward(X_val)
    val_acc = (np.argmax(val_predictions, axis=1) == y_val).mean()

    train_losses.append(epoch_loss)
    val_accs.append(val_acc)
    print(f"Epoch {epoch+1:2d}/{epochs}  loss: {epoch_loss:.4f}  val accuracy: {val_acc*100:.2f}%")

## 6. Watch it learn

In [ ]:
#@title 6. Plot loss and accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, epochs+1), train_losses, 'o-', color='#FF6B4A')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Training loss'); ax1.set_title('Loss'); ax1.grid(True)

ax2.plot(range(1, epochs+1), [a*100 for a in val_accs], 'o-', color='#0E8C82')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Validation accuracy (%)'); ax2.set_title('Accuracy'); ax2.grid(True)

plt.tight_layout()
plt.show()

## 7. The final exam: the test set

We've only used the validation set so far to check progress. Now — and only now — we check performance on data that never influenced a single decision about the model.

In [ ]:
#@title 7. Final test accuracy + confusion matrix
test_predictions = net.forward(X_test)
predicted_labels = np.argmax(test_predictions, axis=1)
test_accuracy = (predicted_labels == y_test).mean()
print(f"Final test accuracy: {test_accuracy*100:.2f}%")

# a confusion matrix, built by hand - no sklearn needed
confusion = np.zeros((10, 10), dtype=int)
for true_label, pred_label in zip(y_test, predicted_labels):
    confusion[true_label, pred_label] += 1

plt.figure(figsize=(7, 6))
plt.imshow(confusion, cmap='Blues')
plt.xticks(range(10)); plt.yticks(range(10))
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix (test set)')
for i in range(10):
    for j in range(10):
        color = 'white' if confusion[i, j] > confusion.max() / 2 else 'black'
        plt.text(j, i, confusion[i, j], ha='center', va='center', color=color, fontsize=9)
plt.colorbar()
plt.show()

print("\nPer-digit accuracy:")
for digit in range(10):
    digit_acc = confusion[digit, digit] / confusion[digit].sum()
    print(f"  {digit}: {digit_acc*100:5.2f}%")

## 8. Now draw your own digit

This is the moment from the slides: "Show It a New Digit — What Does It See?"

Draw a digit below with your mouse, then run the next cell to see the network's confidence for each of the 10 digits. This needs the `ipycanvas` widget — run the install cell once if you don't already have it.

In [ ]:
#@title 8a. Install ipycanvas (run once)
%pip install -q ipycanvas pillow

In [ ]:
#@title 8b. Drawing pad
from ipycanvas import Canvas

canvas = Canvas(width=280, height=280, sync_image_data=True)
canvas.fill_style = 'black'
canvas.fill_rect(0, 0, 280, 280)
canvas.stroke_style = 'white'
canvas.line_width = 18
canvas.line_cap = 'round'

_drawing = {'active': False}

def _on_mouse_down(x, y):
    _drawing['active'] = True
    canvas.begin_path()
    canvas.move_to(x, y)

def _on_mouse_move(x, y):
    if _drawing['active']:
        canvas.line_to(x, y)
        canvas.stroke()

def _on_mouse_up(x, y):
    _drawing['active'] = False

def clear_canvas():
    canvas.fill_style = 'black'
    canvas.fill_rect(0, 0, 280, 280)

canvas.on_mouse_down(_on_mouse_down)
canvas.on_mouse_move(_on_mouse_move)
canvas.on_mouse_up(_on_mouse_up)

print("Draw a digit (0-9) below. Run clear_canvas() in a new cell to start over.")
canvas

## 9. What does it see?

Run this cell after drawing — you can re-run it as many times as you like on the same drawing, or draw something new and run it again.

In [ ]:
#@title 9. predict.py — run on your drawing
from PIL import Image

def canvas_to_input(canvas):
    rgba = canvas.get_image_data()            # 280x280x4
    gray = rgba[:, :, 0].astype(np.uint8)     # white strokes -> high values in the red channel
    small = np.array(Image.fromarray(gray).resize((28, 28), Image.LANCZOS), dtype=np.float64)
    return (small / 255.0).reshape(1, 784)

my_drawing = canvas_to_input(canvas)
probs = net.forward(my_drawing)[0]            # same forward() as everywhere else in this notebook
predicted_digit = np.argmax(probs)

plt.figure(figsize=(3, 3))
plt.imshow(my_drawing.reshape(28, 28), cmap='gray')
plt.title(f"Prediction: {predicted_digit}  ({probs[predicted_digit]*100:.1f}%)")
plt.axis('off')
plt.show()

for digit, p in enumerate(probs):
    bar = '█' * int(p * 40)
    print(f"{digit}: {p*100:5.1f}%  {bar}")